My flow reads saa_attempts.csv, groups by domain, flags domains scoring below 72%, and a person reviews before anything happens.

The agent never changes the question bank or messages a learner without human approval.


In [1]:
# Runs in Google Colab and in a local kernel. Versions match the executed notebook.
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "pandas==2.2.3", "langgraph==1.2.24"])


0

## Step 1. Build a seeded CSV

The file is synthetic. Columns are `learner_id`, `domain`, `task`, `correct` (0 or 1), and `seconds`. Two rows are broken on purpose: one has `correct` set to `yes`, and one has a blank `seconds` value.


In [2]:
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
if (HERE / "saa_flow.ipynb").exists():
    CSV_PATH = HERE / "fixtures" / "saa_attempts.csv"
elif (HERE / "capstone" / "saa_flow.ipynb").exists():
    CSV_PATH = HERE / "capstone" / "fixtures" / "saa_attempts.csv"
else:
    CSV_PATH = HERE / "saa_attempts.csv"
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

SEEDED_ROWS = [
    ("L1", "D1", "1.1", "1", "40"),
    ("L1", "D1", "1.2", "1", "50"),
    ("L1", "D1", "1.3", "0", "60"),
    ("L2", "D1", "1.1", "0", "55"),
    ("L2", "D2", "2.1", "1", "30"),
    ("L2", "D2", "2.2", "1", "35"),
    ("L3", "D2", "2.1", "1", "45"),
    ("L3", "D3", "3.1", "1", "20"),
    ("L3", "D3", "3.2", "0", "80"),
    ("L4", "D4", "4.1", "1", "25"),
    ("L4", "D4", "4.2", "1", "33"),
    ("L4", "D4", "4.3", "1", "41"),
    ("L4", "D4", "4.4", "0", "48"),
    ("L5", "D1", "1.1", "yes", "10"),
    ("L5", "D2", "2.2", "1", ""),
]
frame = pd.DataFrame(SEEDED_ROWS, columns=["learner_id", "domain", "task", "correct", "seconds"])
frame.to_csv(CSV_PATH, index=False)
print(f"wrote {CSV_PATH.name} ({len(frame)} rows)")


wrote saa_attempts.csv (15 rows)


## Step 2. Inspect


In [3]:
raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
print("rows", len(raw), "columns", list(raw.columns))
print(raw.to_string(index=False))


rows 15 columns ['learner_id', 'domain', 'task', 'correct', 'seconds']
learner_id domain task correct seconds
        L1     D1  1.1       1      40
        L1     D1  1.2       1      50
        L1     D1  1.3       0      60
        L2     D1  1.1       0      55
        L2     D2  2.1       1      30
        L2     D2  2.2       1      35
        L3     D2  2.1       1      45
        L3     D3  3.1       1      20
        L3     D3  3.2       0      80
        L4     D4  4.1       1      25
        L4     D4  4.2       1      33
        L4     D4  4.3       1      41
        L4     D4  4.4       0      48
        L5     D1  1.1     yes      10
        L5     D2  2.2       1        


## Step 3. Load and validate

`load_attempts` raises `FileNotFoundError` when the path is missing and `ValueError` when a required column is missing.


In [4]:
REQUIRED_COLUMNS = ["learner_id", "domain", "task", "correct", "seconds"]

def load_attempts(path):
    csv_path = Path(path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"Input CSV not found: {csv_path}")
    loaded = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    missing = [column for column in REQUIRED_COLUMNS if column not in loaded.columns]
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))
    return loaded

loaded = load_attempts(CSV_PATH)
print("loaded", len(loaded), "rows from", CSV_PATH.name)


loaded 15 rows from saa_attempts.csv


## Step 4. Clean

`correct` and `seconds` are converted with `pd.to_numeric(..., errors="coerce")`. Rows that become missing are dropped, and the dropped count is kept.


In [5]:
def clean_attempts(frame):
    cleaned = frame.copy()
    cleaned["correct"] = pd.to_numeric(cleaned["correct"], errors="coerce")
    cleaned["seconds"] = pd.to_numeric(cleaned["seconds"], errors="coerce")
    broken = cleaned[["correct", "seconds"]].isna().any(axis=1)
    dropped_count = int(broken.sum())
    kept = cleaned.loc[~broken].copy()
    return kept, dropped_count

kept, dropped_count = clean_attempts(loaded)
print("kept", len(kept), "dropped_count", dropped_count)


kept 13 dropped_count 2


## Step 5 and Step 6. Group by domain, flag scores below 72%, and define `analyze_saa`

A domain is flagged when its percent correct is strictly below the cutoff. A missing file is caught as `FileNotFoundError` and replaced with a one-row mock. The mock is labeled as a fallback.


In [6]:
MOCK_ATTEMPTS = pd.DataFrame([
    {"learner_id": "mock-1", "domain": "D1", "task": "1.1", "correct": "1", "seconds": "30"},
])

def summarize_domains(kept, threshold):
    summary = []
    for domain, group in kept.groupby("domain", sort=True):
        attempts = int(len(group))
        correct = int(group["correct"].sum())
        percent = round(100 * correct / attempts, 1)
        summary.append({
            "domain": str(domain),
            "attempts": attempts,
            "correct": correct,
            "percent": percent,
        })
    flagged = [row for row in summary if row["percent"] < threshold]
    return summary, flagged

def analyze_saa(path, threshold=72):
    fallback = False
    try:
        loaded_frame = load_attempts(path)
    except FileNotFoundError:
        loaded_frame = MOCK_ATTEMPTS.copy()
        fallback = True
    kept_frame, dropped = clean_attempts(loaded_frame)
    summary, flagged = summarize_domains(kept_frame, threshold)
    return {
        "summary": summary,
        "flagged": flagged,
        "dropped_count": dropped,
        "fallback": fallback,
    }

result = analyze_saa(CSV_PATH, threshold=72)
print("summary", result["summary"])
print("flagged", result["flagged"])
print("dropped_count", result["dropped_count"])


summary [{'domain': 'D1', 'attempts': 4, 'correct': 2, 'percent': 50.0}, {'domain': 'D2', 'attempts': 3, 'correct': 3, 'percent': 100.0}, {'domain': 'D3', 'attempts': 2, 'correct': 1, 'percent': 50.0}, {'domain': 'D4', 'attempts': 4, 'correct': 3, 'percent': 75.0}]
flagged [{'domain': 'D1', 'attempts': 4, 'correct': 2, 'percent': 50.0}, {'domain': 'D3', 'attempts': 2, 'correct': 1, 'percent': 50.0}]
dropped_count 2


## Step 7. Four asserts

These checks print an `ok` line only after the assert passes. `all tests passed` prints only after all four pass.

The fourth check uses cutoff 0. This flow flags percents strictly below the cutoff, so 0 matches no domain. The course names that empty result "a huge threshold flags nothing" for a flow that flags values above a cutoff.


In [7]:
real = analyze_saa(CSV_PATH, threshold=72)
assert real["fallback"] is False
assert real["dropped_count"] == 2
assert real["summary"] == [
    {"domain": "D1", "attempts": 4, "correct": 2, "percent": 50.0},
    {"domain": "D2", "attempts": 3, "correct": 3, "percent": 100.0},
    {"domain": "D3", "attempts": 2, "correct": 1, "percent": 50.0},
    {"domain": "D4", "attempts": 4, "correct": 3, "percent": 75.0},
]
assert real["flagged"] == [
    {"domain": "D1", "attempts": 4, "correct": 2, "percent": 50.0},
    {"domain": "D3", "attempts": 2, "correct": 1, "percent": 50.0},
]
assert set(real) >= {"summary", "flagged", "dropped_count"}
print("ok real numbers")

missing = analyze_saa(CSV_PATH.parent / "saa_attempts_missing.csv", threshold=72)
assert missing["fallback"] is True
assert missing["dropped_count"] == 0
assert missing["summary"] == [{"domain": "D1", "attempts": 1, "correct": 1, "percent": 100.0}]
assert missing["flagged"] == []
print("ok missing file")

missing_column_path = CSV_PATH.parent / "saa_attempts_missing_column.csv"
pd.DataFrame([{"learner_id": "L9", "domain": "D1", "task": "1.1", "correct": "1"}]).to_csv(missing_column_path, index=False)
try:
    analyze_saa(missing_column_path, threshold=72)
    raise AssertionError("missing column did not raise ValueError")
except ValueError as error:
    assert str(error) == "Missing required columns: seconds"
finally:
    missing_column_path.unlink(missing_ok=True)
print("ok missing column")

nothing = analyze_saa(CSV_PATH, threshold=0)
assert nothing["flagged"] == []
print("ok huge threshold flags nothing")
print("all tests passed")


ok real numbers
ok missing file
ok missing column
ok huge threshold flags nothing
all tests passed


## Step 8. LangGraph pause and resume

Nodes run in order: `load → clean → analyze → human_review`. State stores rows as a list of dictionaries, not a DataFrame, because the checkpointer saves plain data. The graph uses `MemorySaver` and `interrupt_before=["human_review"]`. The resume cell calls `update_state` with a decision, then continues.

The decision below is the demo value that lets this notebook finish. It does not change the question bank or message a learner.


In [8]:
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

class ReviewState(TypedDict, total=False):
    path: str
    threshold: float
    rows: list
    summary: list
    flagged: list
    dropped_count: int
    decision: str

def load_node(state: ReviewState) -> dict:
    frame = load_attempts(state["path"])
    rows = frame.to_dict(orient="records")
    return {"rows": rows}

def clean_node(state: ReviewState) -> dict:
    kept_frame, dropped = clean_attempts(pd.DataFrame(state["rows"]))
    rows = kept_frame.to_dict(orient="records")
    return {"rows": rows, "dropped_count": dropped}

def analyze_node(state: ReviewState) -> dict:
    summary, flagged = summarize_domains(pd.DataFrame(state["rows"]), state["threshold"])
    return {"summary": summary, "flagged": flagged}

def human_review(state: ReviewState) -> dict:
    return {"decision": state["decision"]}

builder = StateGraph(ReviewState)
builder.add_node("load", load_node)
builder.add_node("clean", clean_node)
builder.add_node("analyze", analyze_node)
builder.add_node("human_review", human_review)
builder.add_edge(START, "load")
builder.add_edge("load", "clean")
builder.add_edge("clean", "analyze")
builder.add_edge("analyze", "human_review")
builder.add_edge("human_review", END)
graph = builder.compile(checkpointer=MemorySaver(), interrupt_before=["human_review"])

config = {"configurable": {"thread_id": "saa-review-demo"}}
graph.invoke({"path": str(CSV_PATH), "threshold": 72}, config)
snapshot = graph.get_state(config)
print(f"waiting at: {snapshot.next}")
graph.update_state(config, {"decision": "approved by reviewer"})
final = graph.invoke(None, config)
print(f"final decision: {final['decision']}")


waiting at: ('human_review',)
final decision: approved by reviewer


## Limitations

The CSV is synthetic. Those rows are not learner records, and the domain percents are not a measure of skill. The 72 cutoff is the cutoff in the course sentence above. It is not an official conversion of the AWS scaled score of 720. Two rows were built to fail numeric cleaning, so the summary ignores them and reports how many were dropped. The Step 8 decision is a fixed demo value so the notebook can be run from top to bottom. The graph has no step that edits the question bank or contacts a learner.
